In [0]:
-- 1. Read, Type, Trim, and Deduplicate (Batch)
CREATE TEMPORARY LIVE VIEW stg_stores_typed AS
SELECT DISTINCT
  CAST(TRIM(store_id) AS INT) AS store_id,
  CAST(TRIM(region_id) AS INT) AS region_id,
  UPPER(TRIM(store_type)) AS store_type,
  TRIM(store_name) AS store_name,
  TRIM(store_street_address) AS store_street_address,
  TRIM(store_city) AS store_city,
  TRIM(store_state) AS store_state,
  TRIM(store_country) AS store_country,
  TRIM(store_phone) AS store_phone,
  
  -- Handle date normalization
  COALESCE(
    try_to_date(TRIM(first_opened_date), 'M/d/yyyy'),
    try_to_date(TRIM(first_opened_date), 'yyyy-M-d'),
    try_to_date(TRIM(first_opened_date), 'd-MMM-yy'),
    CAST(TRIM(first_opened_date) AS DATE)
  ) AS first_opened_date,
  
  CAST(TRIM(last_remodel_date) AS DATE) AS last_remodel_date,
  CAST(TRIM(total_sqft) AS INT) AS total_sqft,
  CAST(TRIM(grocery_sqft) AS INT) AS grocery_sqft,
  
  ingested_timestamp
FROM mavenmarket.bronze.stores;

-- 2. Run Generic KPI Checks (Including Referential Integrity)
CREATE TEMPORARY LIVE VIEW stg_stores_checked AS
SELECT 
  s.*,
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN s.store_id IS NULL THEN 'Completeness: store_id is null; ' ELSE '' END AS err_store_id,
  
  -- Referential Integrity: Check if region_id exists in the parent table
  CASE WHEN s.region_id IS NOT NULL AND r.region_id IS NULL 
       THEN 'Referential Integrity: region_id not found in regions; ' 
       ELSE '' END AS err_fk_region,
       
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  CASE WHEN s.store_phone IS NULL THEN 'Completeness: store_phone is null; ' ELSE '' END AS warn_phone,
  CASE WHEN s.total_sqft < 0 THEN 'Accuracy: total_sqft is negative; ' ELSE '' END AS warn_sqft_negative,
  CASE WHEN s.grocery_sqft > s.total_sqft THEN 'Consistency: grocery_sqft > total_sqft; ' ELSE '' END AS warn_sqft_logic
FROM stg_stores_typed s
-- Left join to validate the foreign key
LEFT JOIN (SELECT DISTINCT CAST(region_id AS INT) AS region_id FROM mavenmarket.bronze.regions) r 
  ON s.region_id = r.region_id;

-- 3. Consolidate Flags
CREATE TEMPORARY LIVE VIEW stg_stores_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_store_id, err_fk_region)) AS hard_failures,
  TRIM(CONCAT(warn_phone, warn_sqft_negative, warn_sqft_logic)) AS dq_flags
FROM stg_stores_checked;

-- 4. Write Target Tables
CREATE LIVE TABLE silver_stores
COMMENT "Cleaned stores with warn-level DQ flags."
AS SELECT 
  * EXCEPT(err_store_id, err_fk_region, warn_phone, warn_sqft_negative, warn_sqft_logic, hard_failures)
FROM stg_stores_flagged
WHERE hard_failures = '';

CREATE LIVE TABLE quarantine_stores
COMMENT "Quarantined stores failing hard checks"
AS SELECT 
  * EXCEPT(err_store_id, err_fk_region, warn_phone, warn_sqft_negative, warn_sqft_logic, dq_flags),
  hard_failures AS quarantine_reasons
FROM stg_stores_flagged
WHERE hard_failures != '';